In [ ]:
!pip install -q timm

In [ ]:
import kagglehub
import os
import random
import copy
import hashlib
import json
import shutil
import subprocess
import tempfile
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import timm

from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

In [ ]:
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = (not ON_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
print("Environment detected as", "Colab" if ON_COLAB else "Kaggle" if ON_KAGGLE else "local")

if ON_KAGGLE:
    print("Running on Kaggle, dataset is already attached via Input, skipping kagglehub.login().")
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("Loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        print("Could not load HF_TOKEN from Kaggle Secrets (add it under Add-ons > Secrets if needed):", e)
elif os.environ.get("DS_DATA_ROOT"):
    print("Using the data folder from DS_DATA_ROOT, skipping kagglehub.login().")
elif os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    print("Using the Kaggle credentials from the KAGGLE_USERNAME and KAGGLE_KEY environment variables, skipping kagglehub.login().")
else:
    kagglehub.login()

## Configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Force deterministic cuDNN kernels (no algorithm autotuning / benchmarking) so GPU
# convolution ops are reproducible across runs too, not just data loading. Slightly
# slower per epoch than the default benchmarked kernels, but needed for apples-to-apples
# comparisons between runs (see run 4 vs run 8 vs run 9: same config, 3 different
# results, 0.8381 / 0.7631 / 0.7485, before this was added).
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    """Reseed numpy/random inside each DataLoader worker process, so augmentation
    order is reproducible across runs instead of drifting from an OS-derived seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Dataset settings. Change these for a new competition. COMP_SLUG is the Kaggle competition
# name from its URL, TRAIN_SUBDIR holds one folder per class and TEST_SUBDIR holds the test images.
COMP_SLUG = os.environ.get("DS_COMP_SLUG", "your-competition-slug")
TRAIN_SUBDIR = "Train_Set"
TEST_SUBDIR = "Test_Set"

if ON_KAGGLE:
    DATA_ROOT = f"/kaggle/input/competitions/{COMP_SLUG}"
    OUTPUT_DIR = "/kaggle/working"
elif ON_COLAB:
    DATA_ROOT = f"/root/.cache/kagglehub/competitions/{COMP_SLUG}"
    OUTPUT_DIR = "/content"
else:
    # A local machine (JupyterLab on a laptop or a Mac). kagglehub downloads into its cache
    # folder under the home directory unless KAGGLEHUB_CACHE says otherwise, and the outputs
    # go into the current folder.
    _kh_cache = os.environ.get("KAGGLEHUB_CACHE", os.path.join(os.path.expanduser("~"), ".cache", "kagglehub"))
    DATA_ROOT = os.environ.get("DS_DATA_ROOT") or os.path.join(_kh_cache, "competitions", COMP_SLUG)
    OUTPUT_DIR = os.getcwd()
TRAIN_DIR = Path(DATA_ROOT) / TRAIN_SUBDIR
TEST_DIR = Path(DATA_ROOT) / TEST_SUBDIR

# Short label for this run, used in the name of the saved probabilities file so runs
# do not overwrite each other. Change it before every run.
RUN_TAG = "run01-siglip2-b16"

# The default is SigLIP 2 ViT-B/16, the best single recipe of the DATASCAPE 2026 project. It
# normalizes with mean and std 0.5 (see NORM_MEAN and NORM_STD below) and has an attention
# pooling head, so the classifier is the only new layer. For a CLIP, DINOv2 or DINOv3 run change
# MODEL_NAME, MODEL_KWARGS, NORM_MEAN, NORM_STD and LAYER_DECAY together, see the notes next
# to each setting. Take the normalization values from timm.models.get_pretrained_cfg(name).
MODEL_NAME = "vit_base_patch16_siglip_224.v2_webli"
IMG_SIZE = 224
BATCH_SIZE = 32
# An L40 has 46 GB, so the full batch of 32 fits and ACCUM_STEPS = 1. On a 4 GB card use
# BATCH_SIZE = 8 and ACCUM_STEPS = 4, which gives the same effective batch of 32.
ACCUM_STEPS = 1
# Mixed precision on, the L40 has tensor cores. It was off for the GTX 1650 Ti, which has none.
USE_AMP = True
# Added to the per-fold training seed, so the same folds train with different randomness.
# Never change SEED instead, it also seeds the fold split and would break the row alignment.
SEED_OFFSET = 0
# 30 epochs. The cosine schedule follows NUM_EPOCHS, so it still decays to zero at the last epoch.
NUM_EPOCHS = 30
# Peak LR, which applies to the last block and the head. With LAYER_DECAY set, earlier
# blocks get less (0.75 per block step, about 0.02 of the peak at the first block). Without
# LAYER_DECAY it is one LR for the whole model. ConvNeXt needed about 3e-5 to 5e-5 on the DATASCAPE data
# and EfficientNet needed a high LR (3e-4), so check the LR again whenever the architecture changes.
LR = 3e-5
WEIGHT_DECAY = 1e-2

# Extra keyword arguments for timm.create_model. convnext and efficientnet take none, DINOv2
# needs {"img_size": IMG_SIZE} because it is built for 518 px, and CLIP ViT-B/16 needs none.
MODEL_KWARGS = {}

# Normalization constants of the pretrained backbone. These are the SigLIP values. The ImageNet
# values (convnext, efficientnet, DINOv2) are mean [0.485, 0.456, 0.406] and std
# [0.229, 0.224, 0.225], and CLIP uses mean [0.48145466, 0.4578275, 0.40821073] and std
# [0.26862954, 0.26130258, 0.27577711].
NORM_MEAN = [0.5, 0.5, 0.5]
NORM_STD = [0.5, 0.5, 0.5]

# Layer-wise LR decay for ViT models (timm create_optimizer_v2). None disables it and uses
# plain AdamW for the whole model, which is right for convnext. DINOv2, CLIP and SigLIP used 0.75.
LAYER_DECAY = 0.75

# Abort the whole run if the very first epoch of the first fold takes longer than this
# many seconds, to protect the GPU quota from a pathological slowdown (the DataParallel
# attempt took about 2400s for epoch 1, normal on a Kaggle T4 is about 125s including
# warm-up). Set to None to disable the guard.
# Off on this shared server (None), because the guard only protects a Kaggle GPU quota and a
# large model or a busy shared GPU can legitimately take more than 900 s for the first epoch.
FIRST_EPOCH_TIME_LIMIT_S = None

# Stochastic depth (randomly skips residual blocks during training). 0.0 disables it.
# Run 14 used 0.1 and gave no measurable gain over run 13 (0.0), so run 15 goes back to
# 0.0 to differ from run 13 only by the new augmentation below.
DROP_PATH_RATE = 0.0

# Stop a fold's training early if its val macro F1 doesn't improve for this many
# consecutive epochs (saves GPU time once that fold has converged). Set to NUM_EPOCHS
# (or higher) to effectively disable early stopping.
EARLY_STOP_PATIENCE = 30

# Stratified K-fold cross-validation instead of one fixed train/val split. Each fold
# trains its own model and keeps its own best checkpoint; the honest performance
# estimate is the out-of-fold (OOF) score (every training sample scored exactly once,
# by the fold that held it out). The N_FOLDS best checkpoints are then combined into
# a free ensemble for the actual test-set submission. Costs roughly N_FOLDS x the GPU
# time of a single run.
N_FOLDS = 5
SANITY_FOLDS = 2

# Loss function. Focal loss (Lin et al. 2017) down-weights easy/already-correct
# examples so gradient concentrates on hard cases. Run 12 tried gamma=2.0 WITH
# per-class alpha weighting (same weights as class-weighted CE) and got worse OOF
# macro F1 than plain weighted CE (0.7714 vs run 11's 0.7948), Lainnya f1 specifically
# dropped (0.7436 -> 0.6956). Run 13 used gamma alone (alpha=None), relying only on
# WeightedRandomSampler for imbalance handling, and reached OOF macro F1 0.8293.
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0
FOCAL_USE_ALPHA = False

# Mixup/CutMix is no longer implemented (it failed twice, tried at MIXUP_PROB=0.5 and
# 0.15, macro F1 0.7245 and 0.6860). The flag stays only so the launcher cell can assert
# that it is off.
USE_MIXUP = False

# Inference-time TTA. Horizontal flip is always applied; multi-scale (resizing the
# already letterboxed image to each of TTA_SCALES) is only used for the final
# submission if the out-of-fold comparison below shows it actually helps.
# Multi-scale is only used for the submission if it beats single scale on the OOF score. A ViT
# built for 224 px rejects other input sizes, so use [224] for DINOv2, CLIP and SigLIP.
TTA_SCALES = [224]

# Train-to-test shift augmentation. Adversarial validation found that test images differ
# from training images mostly in file size and bytes per pixel (test median 0.126 vs train
# 0.189), so training images are randomly degraded with a downscale then upscale, blur,
# noise and JPEG re-encoding. Each step fires independently with its own probability.
# Only training images are touched, validation and test images are never degraded.
# Set USE_SHIFT_AUG to False to reproduce run 13 behavior. Runs 27 and 33 set it to False.
USE_SHIFT_AUG = False
SHIFT_AUG = dict(
    p_down=0.3, down_scale=(0.4, 0.8),
    p_blur=0.2, blur_radius=(0.3, 1.2),
    p_noise=0.2, noise_std=(2.0, 8.0),
    p_jpeg=0.5, jpeg_quality=(25, 95),
)

# Letterbox every training image to IMG_SIZE once and keep the result in one uint8 array
# on local disk, so each epoch reads small arrays instead of decoding full-size photos
# (data wait was about 48% of epoch time while both GPUs trained). Pixel for pixel this is
# the same image the old pipeline produced, so results do not change. The file is rebuilt
# only when the image list or IMG_SIZE changes. It needs about 1.9 GB of free disk.
USE_IMAGE_CACHE = True
CACHE_DIR = os.environ.get("DS_CACHE_DIR") or tempfile.gettempdir()

# True  = sanity-check mode: tiny per-class subset, SANITY_FOLDS folds, 1 epoch each.
#         Use this to verify the pipeline runs end-to-end (no bugs/shape errors)
#         before spending GPU time.
# False = full run: full data, N_FOLDS folds, NUM_EPOCHS epochs each. Use this for the
#         real training run that produces the model you'll submit.
# Keep this on for the first run on a new machine, it takes a few minutes. Start the notebook
# with the environment variable DS_SANITY=1 for the sanity run, and without it for the real run.
SANITY_CHECK = os.environ.get("DS_SANITY") == "1"
SANITY_SUBSET_PER_CLASS = 20
SANITY_EPOCHS = 1

# Optional config overrides, used by run_queue.py to run several configurations from this one
# notebook. DS_OVERRIDES is a JSON object of config names and values, for example
# {"RUN_TAG": "run50-example", "SEED_OFFSET": 2}. Unknown names stop the run, so a typo cannot
# silently leave a setting at its default.
_overrides = os.environ.get("DS_OVERRIDES")
if _overrides:
    _overrides = json.loads(_overrides)
    _unknown = [k for k in _overrides if k not in globals()]
    assert not _unknown, f"DS_OVERRIDES has names that are not config settings, {_unknown}"
    globals().update(_overrides)
    print("Config overrides from DS_OVERRIDES:", _overrides)

print("Device:", DEVICE)
print("Data root:", DATA_ROOT)
print("Output dir:", OUTPUT_DIR, "| Run tag:", RUN_TAG)


In [ ]:
if ON_KAGGLE:
    path = DATA_ROOT
    print("Running on Kaggle, skipping kagglehub.competition_download(), using attached Input at:", path)
elif (Path(DATA_ROOT) / TRAIN_SUBDIR).exists():
    path = DATA_ROOT
    print("Found the data already at", path, "skipping kagglehub.competition_download().")
else:
    path = kagglehub.competition_download(COMP_SLUG)
    print("Path to competition files:", path)
    if Path(path).resolve() != Path(DATA_ROOT).resolve():
        print("kagglehub used a different folder than DATA_ROOT, switching to", path)
        DATA_ROOT = str(path)
        TRAIN_DIR = Path(DATA_ROOT) / TRAIN_SUBDIR
        TEST_DIR = Path(DATA_ROOT) / TEST_SUBDIR

In [ ]:
for root, dirs, files in os.walk(DATA_ROOT):
    depth = root.replace(DATA_ROOT, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/ ({len(files)} files)")
    if depth >= 2:
        dirs[:] = []

In [ ]:
CLASSES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print("Classes:", CLASSES)

In [ ]:
sample_sub_path = list(Path(DATA_ROOT).glob("*.csv"))[0]
sample_sub = pd.read_csv(sample_sub_path)
print(sample_sub_path)
print(sample_sub.shape)
sample_sub.head()

## Exploratory Data Analysis

In [ ]:
records = []
for cls in CLASSES:
    for fp in sorted((TRAIN_DIR / cls).iterdir()):  # sorted, so the row order (and the folds) match on every machine
        if fp.is_file():
            records.append({"filepath": str(fp), "label": cls})

train_df = pd.DataFrame(records)
print(train_df.shape)
print(train_df["label"].value_counts())

counts = train_df["label"].value_counts().reindex(CLASSES)
plt.figure(figsize=(6, 4))
plt.bar(counts.index, counts.values, color=plt.cm.tab10.colors[:len(CLASSES)])
plt.title("Train class distribution")
plt.ylabel("count")
for i, v in enumerate(counts.values):
    plt.text(i, v + 50, str(v), ha="center")
plt.show()

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(15, 3 * len(CLASSES)))
for row, cls in enumerate(CLASSES):
    sample_paths = train_df[train_df["label"] == cls]["filepath"].sample(5, random_state=SEED)
    for col, fp in enumerate(sample_paths):
        img = Image.open(fp).convert("RGB")
        ax = axes[row, col]
        ax.imshow(img)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(cls)
        ax.set_title(f"{img.size[0]}x{img.size[1]}", fontsize=8)
    axes[row, 0].set_ylabel(cls, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Duplicate check on the training images. The training folders contain byte-identical
# files that carry different labels (the same picture filed under both Aman and Rusak).
# No model can get both copies right, and they pull the loss and the best-epoch pick in
# opposite directions, so by default they are dropped before the folds are built.
# Duplicates that share a label are kept. Set DROP_CONFLICTING_DUPLICATES to False to
# reproduce the run 13 to 15 data.
DROP_CONFLICTING_DUPLICATES = True


def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


with ThreadPoolExecutor(max_workers=8) as pool:
    train_df["md5"] = list(pool.map(file_md5, train_df["filepath"]))

by_hash = train_df.groupby("md5")["label"].agg(["size", "nunique"])
dup_hashes = by_hash.index[by_hash["size"] > 1]
conflict_hashes = by_hash.index[by_hash["nunique"] > 1]
conflict_mask = train_df["md5"].isin(conflict_hashes)

print(f"Identical-file groups: {len(dup_hashes)} covering {int(train_df['md5'].isin(dup_hashes).sum())} images")
print(f"Groups with conflicting labels: {len(conflict_hashes)} covering {int(conflict_mask.sum())} images")
if conflict_mask.any():
    print("Images in conflicting groups per label:", conflict_mask.groupby(train_df["label"]).sum().to_dict())
    for md5_value in conflict_hashes[:5]:
        members = train_df[train_df["md5"] == md5_value]
        print("  e.g.", [(Path(p).name, lab) for p, lab in zip(members["filepath"], members["label"])])

if DROP_CONFLICTING_DUPLICATES and conflict_mask.any():
    train_df = train_df[~conflict_mask].reset_index(drop=True)
    print(f"Dropped {int(conflict_mask.sum())} images, {len(train_df)} remain")
    print(train_df["label"].value_counts().to_dict())
else:
    print("Keeping all images (DROP_CONFLICTING_DUPLICATES is False or nothing conflicts)")

# Put the images in the saved fold order, so the folds match runs 17 to 26 row for row. The
# file system order differs between machines, so without this file the folds would differ.
# fold_order.txt holds one "Class/filename" line per image and sits next to this notebook.
FOLD_ORDER_FILE = "fold_order.txt"
if Path(FOLD_ORDER_FILE).exists():
    saved_order = Path(FOLD_ORDER_FILE).read_text().splitlines()
    rel_names = train_df["filepath"].map(lambda p: f"{Path(p).parent.name}/{Path(p).name}")
    order_pos = {name: i for i, name in enumerate(saved_order)}
    assert set(rel_names) == set(order_pos), (
        f"{FOLD_ORDER_FILE} does not match the training images ({len(set(rel_names) ^ set(order_pos))} names differ)")
    train_df = train_df.iloc[np.argsort(rel_names.map(order_pos).to_numpy(), kind="stable")].reset_index(drop=True)
    print(f"Images reordered to the saved fold order from {FOLD_ORDER_FILE}")
else:
    print(f"No {FOLD_ORDER_FILE} found, the image order is the file system order, so the folds will differ from earlier runs")


## Stratified K-Fold Split

In [ ]:
if SANITY_CHECK:
    # Shuffle, then keep the first SANITY_SUBSET_PER_CLASS rows of each class. The older
    # groupby().apply() version drops the label column in newer pandas.
    fold_df = (
        train_df.sample(frac=1, random_state=SEED)
        .groupby("label", sort=False)
        .head(SANITY_SUBSET_PER_CLASS)
        .reset_index(drop=True)
    )
    folds_to_run = min(SANITY_FOLDS, N_FOLDS)
    EPOCHS_TO_RUN = SANITY_EPOCHS
else:
    fold_df = train_df.reset_index(drop=True)
    folds_to_run = N_FOLDS
    EPOCHS_TO_RUN = NUM_EPOCHS

skf = StratifiedKFold(n_splits=folds_to_run, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(fold_df, fold_df["label"]))

print(f"Sanity check: {SANITY_CHECK} | Folds: {len(fold_splits)} | Epochs/fold: {EPOCHS_TO_RUN} | Samples: {len(fold_df)}")
for i, (tr_idx, va_idx) in enumerate(fold_splits):
    va_counts = fold_df.iloc[va_idx]["label"].value_counts().reindex(CLASSES).to_dict()
    print(f"  fold {i + 1}: train={len(tr_idx)} val={len(va_idx)} val_class_counts={va_counts}")

## Dataset &amp; Transforms

In [ ]:
# NORM_MEAN and NORM_STD come from the config cell and must match the pretrained backbone.


class LetterboxResize:
    """Resize preserving aspect ratio, then pad to a square with the ImageNet mean color."""

    def __init__(self, size, fill=(124, 116, 104)):
        self.size = size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        scale = self.size / max(w, h)
        new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
        img = img.resize((new_w, new_h), Image.BILINEAR)

        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        offset = ((self.size - new_w) // 2, (self.size - new_h) // 2)
        canvas.paste(img, offset)
        return canvas


train_transform = T.Compose([
    LetterboxResize(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

eval_transform = T.Compose([
    LetterboxResize(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


class PackageDataset(Dataset):
    def __init__(self, df, transform, has_labels=True):
        self.filepaths = df["filepath"].tolist()
        self.transform = transform
        self.has_labels = has_labels
        if has_labels:
            self.labels = [CLASS_TO_IDX[l] for l in df["label"]]

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        img = self.transform(img)
        if self.has_labels:
            return img, self.labels[idx]
        return img, self.filepaths[idx]


## Train the Model (5-Fold Cross-Validation)

In [ ]:
# Ensemble + TTA helpers: build one model instance per kept checkpoint, and average
# softmax probabilities across (checkpoint x scale x horizontal-flip) at inference.

def build_ensemble_models(checkpoints):
    models = []
    for f1, state in checkpoints:
        m = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES, **MODEL_KWARGS).to(DEVICE)
        m.load_state_dict(state)
        m.eval()
        models.append(m)
    return models


def predict_proba_tta(models, loader, scales=TTA_SCALES):
    """Return mean softmax probabilities (each row sums to 1) averaged over every
    checkpoint, scale and horizontal flip, plus the loader's second element (labels
    or filepaths) in loader order."""
    n_passes = len(scales) * 2 * len(models)
    all_probs, all_extra = [], []
    with torch.no_grad():
        for images, extra in loader:
            images = images.to(DEVICE, non_blocking=True)
            probs_sum = torch.zeros(images.size(0), NUM_CLASSES, device=DEVICE)

            for scale in scales:
                scaled = images if scale == images.shape[-1] else F.interpolate(
                    images, size=(scale, scale), mode="bilinear", align_corners=False
                )
                for variant in (scaled, torch.flip(scaled, dims=[3])):
                    for m in models:
                        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                            logits = m(variant)
                            probs_sum += torch.softmax(logits.float(), dim=1)

            all_probs.append((probs_sum / n_passes).cpu())
            all_extra.extend(extra.cpu().numpy() if isinstance(extra, torch.Tensor) else extra)

    return torch.cat(all_probs, dim=0), all_extra


In [ ]:
%%writefile img_utils.py
"""Image helpers shared by cv_worker.py and build_cache.py (no torch needed)."""
import io
import random

import numpy as np
from PIL import Image, ImageFilter


class LetterboxResize:
    """Resize preserving aspect ratio, then pad to a square with the ImageNet mean color."""

    def __init__(self, size, fill=(124, 116, 104)):
        self.size = size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        scale = self.size / max(w, h)
        new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
        img = img.resize((new_w, new_h), Image.BILINEAR)

        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        offset = ((self.size - new_w) // 2, (self.size - new_h) // 2)
        canvas.paste(img, offset)
        return canvas


class RandomDegrade:
    """Randomly lower image quality to mimic the compression and resolution differences
    between the training and test photos (test images have a lower median bytes per pixel).
    Each step fires independently with its own probability, so many images stay clean.
    Randomness comes from python's random and numpy, which seed_worker reseeds per worker."""

    def __init__(self, p_down=0.3, down_scale=(0.4, 0.8), p_blur=0.2, blur_radius=(0.3, 1.2),
                 p_noise=0.2, noise_std=(2.0, 8.0), p_jpeg=0.5, jpeg_quality=(25, 95)):
        self.p_down, self.down_scale = p_down, down_scale
        self.p_blur, self.blur_radius = p_blur, blur_radius
        self.p_noise, self.noise_std = p_noise, noise_std
        self.p_jpeg, self.jpeg_quality = p_jpeg, jpeg_quality

    def __call__(self, img):
        w, h = img.size
        if random.random() < self.p_down:
            s = random.uniform(*self.down_scale)
            small = img.resize((max(8, int(w * s)), max(8, int(h * s))), Image.BILINEAR)
            img = small.resize((w, h), Image.BILINEAR)
        if random.random() < self.p_blur:
            img = img.filter(ImageFilter.GaussianBlur(random.uniform(*self.blur_radius)))
        if random.random() < self.p_noise:
            arr = np.asarray(img, dtype=np.float32)
            arr = arr + np.random.normal(0.0, random.uniform(*self.noise_std), arr.shape)
            img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
        if random.random() < self.p_jpeg:
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=random.randint(int(self.jpeg_quality[0]), int(self.jpeg_quality[1])))
            buf.seek(0)
            img = Image.open(buf).convert("RGB")
        return img

In [ ]:
%%writefile build_cache.py
"""Resize every listed image once to the letterboxed square size and store them in one
uint8 .npy file, so training reads small arrays instead of decoding the full-size photos."""
import argparse
import warnings
from multiprocessing import Pool
from pathlib import Path

import numpy as np
from PIL import Image

from img_utils import LetterboxResize

warnings.filterwarnings("ignore", category=UserWarning, module="PIL")


def load_one(args):
    path, size = args
    with Image.open(path) as im:
        return np.asarray(LetterboxResize(size)(im.convert("RGB")), dtype=np.uint8)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--paths-file", required=True)
    parser.add_argument("--out", required=True)
    parser.add_argument("--size", type=int, required=True)
    parser.add_argument("--procs", type=int, default=4)
    args = parser.parse_args()

    paths = Path(args.paths_file).read_text().splitlines()
    tmp_out = args.out + ".partial.npy"
    out = np.lib.format.open_memmap(tmp_out, mode="w+", dtype=np.uint8, shape=(len(paths), args.size, args.size, 3))
    with Pool(args.procs) as pool:
        for i, arr in enumerate(pool.imap(load_one, [(p, args.size) for p in paths], chunksize=32)):
            out[i] = arr
            if i % 2000 == 0:
                print(f"  cached {i}/{len(paths)}", flush=True)
    out.flush()
    del out
    Path(tmp_out).rename(args.out)  # renamed last, so a half-written cache is never picked up
    print(f"cached {len(paths)} images to {args.out}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile cv_worker.py
"""Standalone fold trainer. The notebook launches one of these per GPU as a subprocess."""
import argparse
import copy
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import f1_score
from timm.optim import create_optimizer_v2
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

from img_utils import LetterboxResize, RandomDegrade

warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_transforms(img_size, cached=False, aug=None, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """cached=True means the images were already letterboxed to img_size by build_cache.py.
    aug is a dict of RandomDegrade settings (or None) applied to training images only.
    mean and std are the normalization constants of the pretrained backbone."""
    pre = [] if cached else [LetterboxResize(img_size)]
    degrade = [RandomDegrade(**aug)] if aug else []
    train_transform = T.Compose(pre + degrade + [
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    eval_transform = T.Compose(pre + [
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    return train_transform, eval_transform


class PackageDataset(Dataset):
    def __init__(self, df, transform, class_to_idx):
        self.filepaths = df["filepath"].tolist()
        self.transform = transform
        self.labels = [class_to_idx[label] for label in df["label"]]

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]


class CachedDataset(Dataset):
    """Reads already letterboxed images from the uint8 cache array built by build_cache.py.
    positions index the cache (they are the fold_df row numbers)."""

    def __init__(self, cache, positions, labels, transform):
        self.cache = cache
        self.positions = positions
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, idx):
        img = Image.fromarray(np.asarray(self.cache[self.positions[idx]]))
        return self.transform(img), int(self.labels[idx])


class FocalLoss(nn.Module):
    """Focal loss (Lin et al. 2017), same as the notebook version."""

    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, weight=self.alpha,
            label_smoothing=self.label_smoothing, reduction="none",
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None, use_amp=True, accum_steps=1):
    """One pass over the loader. Returns avg loss, macro F1, targets, predictions and a
    timing dict with the epoch seconds and the seconds spent waiting on the DataLoader.
    With accum_steps above 1 the optimizer steps once per accum_steps batches, so a small
    batch can reproduce a larger one (the model has no batch norm, so the gradient matches)."""
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds, all_targets = [], []
    data_wait = 0.0
    epoch_start = time.perf_counter()
    fetch_start = epoch_start
    use_amp = use_amp and device.type == "cuda"
    n_batches = len(loader)
    if is_train:
        optimizer.zero_grad(set_to_none=True)

    with torch.set_grad_enabled(is_train):
        for batch_idx, (images, targets) in enumerate(loader):
            data_wait += time.perf_counter() - fetch_start

            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            if is_train:
                group_size = min(accum_steps, n_batches - (batch_idx // accum_steps) * accum_steps)
                with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                    outputs = model(images)
                    loss = criterion(outputs, targets)
                scaler.scale(loss / group_size).backward()
                if (batch_idx + 1) % accum_steps == 0 or batch_idx + 1 == n_batches:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            else:
                with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                    outputs = model(images)
                    loss = criterion(outputs, targets)

            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            total_loss += loss.item() * images.size(0)
            fetch_start = time.perf_counter()

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    timing = {"seconds": time.perf_counter() - epoch_start, "data_wait": data_wait}
    return avg_loss, macro_f1, all_targets, all_preds, timing


def predict_proba_tta(model, loader, scales, device, num_classes, use_amp=True):
    """Mean softmax probabilities over the given scales and horizontal flip."""
    n_passes = len(scales) * 2
    all_probs = []
    use_amp = use_amp and device.type == "cuda"
    model.eval()
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device, non_blocking=True)
            probs_sum = torch.zeros(images.size(0), num_classes, device=device)
            for scale in scales:
                scaled = images if scale == images.shape[-1] else F.interpolate(
                    images, size=(scale, scale), mode="bilinear", align_corners=False
                )
                for variant in (scaled, torch.flip(scaled, dims=[3])):
                    with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                        logits = model(variant)
                    probs_sum += torch.softmax(logits.float(), dim=1)
            all_probs.append((probs_sum / n_passes).cpu())
    return torch.cat(all_probs, dim=0)


def train_one_fold(fold_idx, train_idx, val_idx, fold_df, cfg, out_dir, device,
                   is_first_fold, n_folds_for_worker):
    # Reseed per fold so a fold gives the same result whichever worker or order runs it.
    seed = cfg["SEED"] + fold_idx + cfg.get("SEED_OFFSET", 0)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    classes = cfg["CLASSES"]
    num_classes = len(classes)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    epochs = cfg["EPOCHS_TO_RUN"]
    use_amp = cfg.get("USE_AMP", True)
    accum_steps = cfg.get("ACCUM_STEPS", 1)
    # Every log line carries this tag so lines from the two GPU workers stay readable.
    tag = f"[fold {fold_idx + 1}/{cfg['n_folds_total']}]"
    print(
        f"{tag} started, batch {cfg['BATCH_SIZE']} x accumulation {accum_steps} = {cfg['BATCH_SIZE'] * accum_steps}, "
        f"mixed precision {'on' if use_amp and device.type == 'cuda' else 'off'}, seed offset {cfg.get('SEED_OFFSET', 0)}",
        flush=True,
    )

    cache = np.load(cfg["CACHE_PATH"], mmap_mode="r") if cfg.get("CACHE_PATH") else None
    train_transform, eval_transform = build_transforms(
        cfg["IMG_SIZE"], cached=cache is not None, aug=cfg.get("SHIFT_AUG"),
        mean=cfg.get("NORM_MEAN", IMAGENET_MEAN), std=cfg.get("NORM_STD", IMAGENET_STD),
    )
    fold_train_df = fold_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = fold_df.iloc[val_idx].reset_index(drop=True)
    if cache is not None:
        all_labels = np.array([class_to_idx[label] for label in fold_df["label"]])
        train_dataset = CachedDataset(cache, train_idx, all_labels[train_idx], train_transform)
        val_dataset = CachedDataset(cache, val_idx, all_labels[val_idx], eval_transform)
    else:
        train_dataset = PackageDataset(fold_train_df, train_transform, class_to_idx)
        val_dataset = PackageDataset(fold_val_df, eval_transform, class_to_idx)
    print(
        f"{tag} data from {'image cache' if cache is not None else 'original files'}, "
        f"shift augmentation {'on' if cfg.get('SHIFT_AUG') else 'off'}",
        flush=True,
    )

    class_counts = fold_train_df["label"].value_counts().reindex(classes)
    class_weights = (class_counts.sum() / (num_classes * class_counts)).values
    sample_weights = fold_train_df["label"].map(dict(zip(classes, class_weights))).values

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=torch.Generator().manual_seed(seed),
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg["BATCH_SIZE"],
        sampler=sampler,
        num_workers=cfg["NUM_WORKERS"],
        pin_memory=device.type == "cuda",
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(seed),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg["BATCH_SIZE"],
        shuffle=False,
        num_workers=cfg["NUM_WORKERS"],
        pin_memory=device.type == "cuda",
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(seed),
    )

    model = timm.create_model(
        cfg["MODEL_NAME"], pretrained=True, num_classes=num_classes,
        drop_path_rate=cfg["DROP_PATH_RATE"], **cfg.get("MODEL_KWARGS", {}),
    ).to(device)

    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
    if cfg["USE_FOCAL_LOSS"]:
        alpha = class_weights_tensor if cfg["FOCAL_USE_ALPHA"] else None
        criterion = FocalLoss(alpha=alpha, gamma=cfg["FOCAL_GAMMA"], label_smoothing=0.1)
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
    layer_decay = cfg.get("LAYER_DECAY")
    if layer_decay:
        # Layer-wise LR decay for ViTs, earlier blocks get a smaller LR. timm only stores the
        # decay as a per-group lr_scale that its own schedulers apply, so set each group's lr here
        # for torch's scheduler, which keeps the ratios between groups. Bias and norm weights get
        # no weight decay in this branch.
        optimizer = create_optimizer_v2(
            model, opt="adamw", lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"], layer_decay=layer_decay
        )
        for group in optimizer.param_groups:
            group["lr"] = cfg["LR"] * group.get("lr_scale", 1.0)
    else:
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"]
        )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(device="cuda", enabled=(device.type == "cuda" and use_amp))

    best_val_f1 = 0.0
    best_state = None
    best_epoch = 0
    best_targets, best_preds = None, None
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_f1, _, _, train_timing = run_epoch(
            model, train_loader, criterion, device, optimizer, scaler, use_amp, accum_steps
        )
        val_loss, val_f1, val_targets, val_preds, val_timing = run_epoch(
            model, val_loader, criterion, device
        )
        scheduler.step()

        epoch_seconds = train_timing["seconds"] + val_timing["seconds"]
        wait_pct = 100 * (train_timing["data_wait"] + val_timing["data_wait"]) / epoch_seconds
        print(
            f"{tag} Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} val_f1={val_f1:.4f} | "
            f"{epoch_seconds:.0f}s (data wait {wait_pct:.0f}%)",
            flush=True,
        )

        limit = cfg["FIRST_EPOCH_TIME_LIMIT_S"]
        if is_first_fold and epoch == 1 and limit and epoch_seconds > limit:
            raise RuntimeError(
                f"First epoch took {epoch_seconds:.0f}s, over the {limit}s guard. "
                f"Aborting to protect the GPU quota (set FIRST_EPOCH_TIME_LIMIT_S = None to disable)."
            )
        if is_first_fold and epoch == 2:
            est_hours = epoch_seconds * epochs * n_folds_for_worker / 3600
            print(
                f"{tag} Rough estimate for this worker at this speed, about {est_hours:.1f} hours of wall time",
                flush=True,
            )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_targets, best_preds = val_targets, val_preds
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= cfg["EARLY_STOP_PATIENCE"]:
                print(
                    f"{tag} Early stopping, no improvement for {cfg['EARLY_STOP_PATIENCE']} epochs",
                    flush=True,
                )
                break

    print(f"{tag} best val macro F1 {best_val_f1:.4f} at epoch {best_epoch}", flush=True)

    model.load_state_dict(best_state)
    probs_flip = predict_proba_tta(model, val_loader, [cfg["IMG_SIZE"]], device, num_classes, use_amp)
    probs_tta = predict_proba_tta(model, val_loader, cfg["TTA_SCALES"], device, num_classes, use_amp)

    torch.save({k: v.detach().cpu() for k, v in best_state.items()}, out_dir / f"fold{fold_idx}_ckpt.pt")
    np.savez_compressed(
        out_dir / f"fold{fold_idx}_oof.npz",
        val_idx=np.asarray(val_idx),
        targets=np.asarray(best_targets),
        preds_single=np.asarray(best_preds),
        probs_flip=probs_flip.numpy(),
        probs_tta=probs_tta.numpy(),
    )
    # The summary file is written last and doubles as the "this fold finished" marker.
    (out_dir / f"fold{fold_idx}_summary.json").write_text(json.dumps({
        "fold": fold_idx,
        "best_val_f1": float(best_val_f1),
        "best_epoch": int(best_epoch),
    }))

    del model, optimizer, scheduler, scaler, best_state
    torch.cuda.empty_cache()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--work-dir", required=True)
    parser.add_argument("--folds", required=True)
    args = parser.parse_args()

    work_dir = Path(args.work_dir)
    cfg = json.loads((work_dir / "cfg.json").read_text())
    fold_df = pd.read_csv(work_dir / "fold_df.csv")
    splits = np.load(work_dir / "splits.npz")
    folds = [int(f) for f in args.folds.split(",")]

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else device.type
    print(f"Worker started on {gpu_name} for folds {[f + 1 for f in folds]}", flush=True)

    for i, fold_idx in enumerate(folds):
        train_one_fold(
            fold_idx,
            splits[f"train_{fold_idx}"],
            splits[f"val_{fold_idx}"],
            fold_df,
            cfg,
            work_dir,
            device,
            is_first_fold=(i == 0),
            n_folds_for_worker=len(folds),
        )
    print("Worker finished", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# Settings for this cell. PARALLEL_FOLDS runs one training process per visible GPU, each
# pinned to its own GPU through CUDA_VISIBLE_DEVICES and given a share of the folds, so
# the two Kaggle T4s train different folds at the same time with no cross-GPU traffic.
# With a single GPU (or False) one process trains all folds in order. NUM_WORKERS is the
# DataLoader worker count per process, so 2 processes x 6 workers uses 12 of the lab server's 80 CPUs.
PARALLEL_FOLDS = True
NUM_WORKERS = 6
# True keeps the folds that already finished in this run's folder and trains only the rest, so
# a run interrupted by sleep or a crash continues. A fold interrupted halfway starts again from
# epoch 1. The settings must be unchanged, otherwise it stops with a message.
RESUME = True

assert not USE_MIXUP, "cv_worker.py does not implement mixup/cutmix, set USE_MIXUP = False"
for needed in ("cv_worker.py", "img_utils.py", "build_cache.py"):
    assert Path(needed).exists(), f"Run the {needed} cell above first"

work_dir = Path(OUTPUT_DIR) / f"cv-{RUN_TAG}{'-sanity' if SANITY_CHECK else ''}"
work_dir.mkdir(parents=True, exist_ok=True)
if not RESUME:
    for stale in work_dir.glob("*"):
        if stale.is_file():
            stale.unlink()

# One-time image cache, see USE_IMAGE_CACHE in the config cell. The file name carries a
# hash of the image list and IMG_SIZE, so a changed list or size builds a fresh cache.
cache_path = None
if USE_IMAGE_CACHE:
    paths_text = "\n".join(fold_df["filepath"])
    cache_key = hashlib.md5(f"{IMG_SIZE}\n{paths_text}".encode()).hexdigest()[:10]
    cache_path = Path(CACHE_DIR) / f"imgcache_{IMG_SIZE}_{cache_key}.npy"
    if cache_path.exists():
        print("Using existing image cache", cache_path)
    else:
        need_gb = len(fold_df) * IMG_SIZE * IMG_SIZE * 3 * 1.1 / 1e9
        free_gb = shutil.disk_usage(CACHE_DIR).free / 1e9
        assert free_gb > need_gb, f"Image cache needs about {need_gb:.1f} GB in {CACHE_DIR}, only {free_gb:.1f} GB free"
        (work_dir / "cache_paths.txt").write_text(paths_text)
        print(f"Building image cache ({need_gb:.1f} GB, one time per session)...")
        build_start = time.time()
        build = subprocess.run(
            [sys.executable, "build_cache.py", "--paths-file", str(work_dir / "cache_paths.txt"),
             "--out", str(cache_path), "--size", str(IMG_SIZE), "--procs", str(min(12, os.cpu_count() or 2))],
            capture_output=True, text=True,
        )
        print(build.stdout[-500:])
        if build.returncode != 0:
            raise RuntimeError("build_cache.py failed\n" + build.stderr[-2000:])
        print(f"Image cache built in {time.time() - build_start:.0f}s")

cfg = {
    "SEED": SEED, "CLASSES": CLASSES, "IMG_SIZE": IMG_SIZE, "EPOCHS_TO_RUN": EPOCHS_TO_RUN,
    "BATCH_SIZE": BATCH_SIZE, "NUM_WORKERS": NUM_WORKERS, "MODEL_NAME": MODEL_NAME,
    "MODEL_KWARGS": MODEL_KWARGS, "LAYER_DECAY": LAYER_DECAY,
    "NORM_MEAN": NORM_MEAN, "NORM_STD": NORM_STD,
    "USE_AMP": USE_AMP, "ACCUM_STEPS": ACCUM_STEPS, "SEED_OFFSET": SEED_OFFSET,
    "DROP_PATH_RATE": DROP_PATH_RATE, "USE_FOCAL_LOSS": USE_FOCAL_LOSS,
    "FOCAL_USE_ALPHA": FOCAL_USE_ALPHA, "FOCAL_GAMMA": FOCAL_GAMMA, "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY, "EARLY_STOP_PATIENCE": EARLY_STOP_PATIENCE,
    "TTA_SCALES": TTA_SCALES, "FIRST_EPOCH_TIME_LIMIT_S": FIRST_EPOCH_TIME_LIMIT_S,
    "n_folds_total": len(fold_splits),
    "CACHE_PATH": str(cache_path) if cache_path else None,
    "SHIFT_AUG": SHIFT_AUG if USE_SHIFT_AUG else None,
}
cfg_text = json.dumps(cfg)
if RESUME and (work_dir / "cfg.json").exists():
    assert json.loads((work_dir / "cfg.json").read_text()) == json.loads(cfg_text), (
        "RESUME is on but the settings differ from the earlier run in this folder, change RUN_TAG or set RESUME = False")
(work_dir / "cfg.json").write_text(cfg_text)
fold_df.to_csv(work_dir / "fold_df.csv", index=False)
split_arrays = {}
for i, (tr_idx, va_idx) in enumerate(fold_splits):
    split_arrays[f"train_{i}"] = tr_idx
    split_arrays[f"val_{i}"] = va_idx
np.savez(work_dir / "splits.npz", **split_arrays)

done_folds = [i for i in range(len(fold_splits)) if RESUME and (work_dir / f"fold{i}_summary.json").exists()]
remaining = [i for i in range(len(fold_splits)) if i not in done_folds]
if done_folds:
    print(f"RESUME is on, folds already finished and skipped {[f + 1 for f in done_folds]}")
n_gpus = torch.cuda.device_count()
n_procs = n_gpus if (PARALLEL_FOLDS and n_gpus > 1) else 1
n_procs = max(1, min(n_procs, len(remaining)))
assignments = [a for a in (remaining[w::n_procs] for w in range(n_procs)) if a]
# Optional manual split for GPUs of different speed (a shared server). DS_FOLD_ASSIGNMENT lists the
# 1-based folds of each worker, workers separated by ";", for example "1,3,5;2;4" gives worker 0
# folds 1, 3 and 5, worker 1 fold 2 and worker 2 fold 4. Worker w runs on the w-th GPU of
# CUDA_VISIBLE_DEVICES. Folds that already finished are skipped, the rest must be covered once.
_manual = os.environ.get("DS_FOLD_ASSIGNMENT")
if _manual:
    assignments = [[int(f) - 1 for f in group.split(",") if f.strip() and int(f) - 1 in remaining] for group in _manual.split(";")]
    assignments = [a for a in assignments if a]
    assert sorted(f for a in assignments for f in a) == sorted(remaining), (
        f"DS_FOLD_ASSIGNMENT must cover every remaining fold {[f + 1 for f in remaining]} exactly once")
    assert len(assignments) <= max(1, n_gpus), f"DS_FOLD_ASSIGNMENT has {len(assignments)} workers but only {n_gpus} GPU(s) are visible"
n_procs = len(assignments)
print(f"{n_gpus} GPU(s) visible, {n_procs} training process(es), fold assignment {[[f + 1 for f in fl] for fl in assignments]}")
print(f"model {MODEL_NAME} | LR {LR} | layer decay {LAYER_DECAY} | drop_path_rate={DROP_PATH_RATE} | "
      f"shift augmentation {'on' if USE_SHIFT_AUG else 'off'} | "
      f"image cache {'on' if cache_path else 'off'} | logs and results in {work_dir}")

worker_path = str(Path("cv_worker.py").resolve())
workers = []
for w, folds in enumerate(assignments):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    if n_procs > 1:
        # Pin worker w to the w-th GPU of the parent's visible list, so a CUDA_VISIBLE_DEVICES
        # set before starting the notebook (for example "1,2" on a shared server) is respected.
        _visible = [g.strip() for g in os.environ.get("CUDA_VISIBLE_DEVICES", "").split(",") if g.strip()]
        env["CUDA_VISIBLE_DEVICES"] = _visible[w] if _visible else str(w)
    log_path = work_dir / f"worker{w}.log"
    log_file = open(log_path, "w")
    proc = subprocess.Popen(
        [sys.executable, worker_path, "--work-dir", str(work_dir), "--folds", ",".join(map(str, folds))],
        env=env, stdout=log_file, stderr=subprocess.STDOUT,
    )
    workers.append({"proc": proc, "log_path": log_path, "log_file": log_file, "pos": 0})


def drain_logs(final=False):
    """Print any new complete log lines from each worker (all remaining text if final)."""
    for w, info in enumerate(workers):
        with open(info["log_path"], "rb") as f:
            f.seek(info["pos"])
            data = f.read()
        cut = len(data) if final else data.rfind(b"\n") + 1
        info["pos"] += cut
        for line in data[:cut].decode(errors="replace").splitlines():
            print(f"[gpu{w}] {line}" if n_procs > 1 else line)


try:
    while True:
        drain_logs()
        codes = [info["proc"].poll() for info in workers]
        if any(c not in (None, 0) for c in codes) or all(c == 0 for c in codes):
            break
        time.sleep(5)
finally:
    # Also runs if the cell is interrupted, so no orphan process keeps a GPU busy.
    for info in workers:
        if info["proc"].poll() is None:
            info["proc"].kill()
        info["proc"].wait()
        info["log_file"].close()
    drain_logs(final=True)

exit_codes = [info["proc"].returncode for info in workers]
if any(c != 0 for c in exit_codes):
    raise RuntimeError(f"Training process(es) failed with exit codes {exit_codes}, see the log lines above")

all_fold_checkpoints = []  # (val_f1, state_dict, fold_idx) for each fold's best checkpoint
fold_results = []  # per-fold summary: fold, best_val_f1, best_epoch

oof_targets = np.full(len(fold_df), -1, dtype=int)
oof_preds_single = np.full(len(fold_df), -1, dtype=int)  # no TTA, straight from run_epoch
oof_probs_flip = np.zeros((len(fold_df), NUM_CLASSES), dtype=np.float32)  # single scale + hflip
oof_probs_tta = np.zeros((len(fold_df), NUM_CLASSES), dtype=np.float32)  # multi-scale + hflip

for fold_idx in range(len(fold_splits)):
    summary = json.loads((work_dir / f"fold{fold_idx}_summary.json").read_text())
    oof = np.load(work_dir / f"fold{fold_idx}_oof.npz")
    val_idx = oof["val_idx"]
    oof_targets[val_idx] = oof["targets"]
    oof_preds_single[val_idx] = oof["preds_single"]
    oof_probs_flip[val_idx] = oof["probs_flip"]
    oof_probs_tta[val_idx] = oof["probs_tta"]
    state = torch.load(work_dir / f"fold{fold_idx}_ckpt.pt", map_location="cpu")
    all_fold_checkpoints.append((summary["best_val_f1"], state, fold_idx))
    fold_results.append(summary)

assert (oof_targets >= 0).all(), "Some samples never got an out-of-fold prediction"

fold_f1s = np.array([r["best_val_f1"] for r in fold_results])
print(f"\n{'=' * 20} K-Fold Summary {'=' * 20}")
for r in fold_results:
    print(f"  fold {r['fold'] + 1}: val macro F1 = {r['best_val_f1']:.4f} (best epoch {r['best_epoch']})")
print(f"Mean per-fold macro F1: {fold_f1s.mean():.4f} +/- {fold_f1s.std():.4f}")

topk_checkpoints = [(f1, state) for f1, state, _ in all_fold_checkpoints]
topk_checkpoints.sort(key=lambda x: x[0], reverse=True)


In [ ]:
ensemble_models = build_ensemble_models(topk_checkpoints)
print(f"Built ensemble of {len(ensemble_models)} fold checkpoint(s), TTA scales {TTA_SCALES} x hflip "
      f"= {len(ensemble_models) * len(TTA_SCALES) * 2} forward passes per image.")

## Validation Report (Out-of-Fold)

In [ ]:
print("=== Out-of-fold (OOF): single best checkpoint per fold, no TTA ===")
print(classification_report(oof_targets, oof_preds_single, target_names=CLASSES, digits=4))

oof_preds_flip = oof_probs_flip.argmax(axis=1)
print("=== Out-of-fold (OOF): per-fold checkpoint, single scale + hflip ===")
print(classification_report(oof_targets, oof_preds_flip, target_names=CLASSES, digits=4))

oof_preds_tta = oof_probs_tta.argmax(axis=1)
print("=== Out-of-fold (OOF): per-fold checkpoint, multi-scale + hflip ===")
print(classification_report(oof_targets, oof_preds_tta, target_names=CLASSES, digits=4))

no_tta_f1 = f1_score(oof_targets, oof_preds_single, average="macro")
single_f1 = f1_score(oof_targets, oof_preds_flip, average="macro")
tta_f1 = f1_score(oof_targets, oof_preds_tta, average="macro")
print(f"OOF macro F1: no TTA={no_tta_f1:.4f} | single scale + hflip={single_f1:.4f} | "
      f"multi-scale + hflip={tta_f1:.4f} | "
      f"multi-scale {'IMPROVED' if tta_f1 > single_f1 else 'did not improve'} by {tta_f1 - single_f1:+.4f}")
print(f"Mean per-fold macro F1 was {fold_f1s.mean():.4f} +/- {fold_f1s.std():.4f}; "
      f"OOF pools every fold's held-out predictions into one score and is the more reliable overall estimate.")

USE_TTA_FOR_SUBMISSION = tta_f1 > single_f1
final_oof_preds = oof_preds_tta if USE_TTA_FOR_SUBMISSION else oof_preds_flip

cm = confusion_matrix(oof_targets, final_oof_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES)
ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(im)
plt.title(f"OOF Confusion Matrix ({'multi-scale + hflip' if USE_TTA_FOR_SUBMISSION else 'single scale + hflip'})")
plt.tight_layout()
plt.show()

In [ ]:
# Label-issue scan on the out-of-fold probabilities (confident learning with cleanlab).
# This only reports and saves a csv, it never changes the training data. Most flagged
# images are the model's confident mistakes and not wrong labels, so review before
# removing anything. Needs internet once to pip install cleanlab. Set to False to skip.
RUN_LABEL_ISSUE_SCAN = True

if RUN_LABEL_ISSUE_SCAN:
    try:
        from cleanlab.filter import find_label_issues
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cleanlab"], check=False)
        try:
            from cleanlab.filter import find_label_issues
        except ImportError:
            find_label_issues = None
            print("cleanlab could not be installed, skipping the label-issue scan")

    if find_label_issues is not None:
        issue_idx = find_label_issues(
            labels=oof_targets, pred_probs=oof_probs_flip,
            filter_by="prune_by_noise_rate", return_indices_ranked_by="self_confidence", n_jobs=1,
        )
        oof_pred = oof_probs_flip.argmax(axis=1)
        issues = pd.DataFrame({
            "filepath": fold_df["filepath"].values[issue_idx],
            "given": [CLASSES[i] for i in oof_targets[issue_idx]],
            "predicted": [CLASSES[i] for i in oof_pred[issue_idx]],
            "p_given": oof_probs_flip[issue_idx, oof_targets[issue_idx]].round(4),
        })
        issues_path = Path(OUTPUT_DIR) / f"label_issues-{RUN_TAG}.csv"
        issues.to_csv(issues_path, index=False)

        class_totals = np.bincount(oof_targets, minlength=len(CLASSES))
        print(f"Flagged {len(issues)} of {len(fold_df)} images ({100 * len(issues) / len(fold_df):.1f}%), saved to {issues_path}")
        for c, name in enumerate(CLASSES):
            n_flag = int((issues["given"] == name).sum())
            print(f"  given {name}: {n_flag} of {class_totals[c]} flagged")
        print(issues.groupby(["given", "predicted"]).size().rename("count").reset_index().to_string(index=False))

        # Look at the most suspicious images of each given label (lowest probability for the given label).
        n_show = 4
        fig, axes = plt.subplots(len(CLASSES), n_show, figsize=(4 * n_show, 4 * len(CLASSES)))
        for row, name in enumerate(CLASSES):
            top = issues[issues["given"] == name].head(n_show)
            for col in range(n_show):
                ax = axes[row, col]
                ax.axis("off")
                if col < len(top):
                    r = top.iloc[col]
                    ax.imshow(Image.open(r["filepath"]).convert("RGB"))
                    ax.set_title(f"given {r['given']}, pred {r['predicted']}, p={r['p_given']:.2f}", fontsize=8)
        plt.tight_layout()
        plt.show()

## Inference on Test Set

In [ ]:
test_filepaths = sorted([str(p) for p in TEST_DIR.iterdir() if p.is_file()])
test_df = pd.DataFrame({"filepath": test_filepaths})
print("Test images:", len(test_df))

test_dataset = PackageDataset(test_df, eval_transform, has_labels=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=DEVICE.type == "cuda",
    worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED),
)

test_scales = TTA_SCALES if USE_TTA_FOR_SUBMISSION else [IMG_SIZE]
print(f"Using {len(ensemble_models)}-fold ensemble, scales={test_scales}, hflip=True "
      f"(multi-scale TTA {'validated as better' if USE_TTA_FOR_SUBMISSION else 'did not beat single-scale'} "
      f"via OOF: {tta_f1:.4f} vs {single_f1:.4f})")

test_probs, all_filepaths = predict_proba_tta(ensemble_models, test_loader, scales=test_scales)
all_preds = test_probs.argmax(dim=1).numpy()
all_filenames = [Path(fp).name for fp in all_filepaths]

pred_labels = [IDX_TO_CLASS[p] for p in all_preds]
print(pd.Series(pred_labels).value_counts())

## Generate Submission

In [ ]:
submission_df = pd.DataFrame({"filename": all_filenames, "label": pred_labels})

assert set(submission_df["filename"]) == set(sample_sub["filename"]), \
    "Submission filenames do not match sample_submission.csv"

submission_df = submission_df.set_index("filename").loc[sample_sub["filename"]].reset_index()

submission_path = f"{OUTPUT_DIR}/submission.csv"
submission_df.to_csv(submission_path, index=False)
tagged_path = f"{OUTPUT_DIR}/submission-{RUN_TAG}.csv"
submission_df.to_csv(tagged_path, index=False)
print("Saved:", submission_path, "and", tagged_path)
submission_df.head()

In [ ]:
# Save everything needed to re-tune or blend predictions later without retraining.
# oof_* arrays are one row per training sample (fold_df order, scored by the one fold that
# held it out). test_probs are the full-ensemble probabilities in test_filenames order.
probs_path = f"{OUTPUT_DIR}/probs-{RUN_TAG}.npz"
np.savez_compressed(
    probs_path,
    classes=np.array(CLASSES),
    oof_filepaths=fold_df["filepath"].to_numpy().astype(str),
    oof_targets=oof_targets,
    oof_probs_flip=oof_probs_flip,
    oof_probs_tta=oof_probs_tta,
    test_filenames=np.array(all_filenames),
    test_probs=test_probs.numpy(),
    test_scales=np.array(test_scales),
)
print("Saved:", probs_path)